In [33]:
import numpy as np
import pandas as pd
import os
from matplotlib import pyplot as plt
from scipy.fft import fft, fftfreq
from scipy import signal
%matplotlib qt
import os
from os import listdir
from os.path import isfile, join
import re

In [60]:
def PSDanalysis(list2, list3, t):
    # Convert to numpy arrays
    list2 = np.asarray(list2)
    list3 = np.asarray(list3)

    # Assume 1D, get first column
    list2_vals = list2[:, 0] if list2.ndim > 1 else list2
    list3_vals = list3[:, 0] if list3.ndim > 1 else list3

    S = np.max(list3_vals) + np.min(list3_vals)
    D = np.max(list3_vals) - np.min(list3_vals)

    x = ((2 * list2_vals) - S) / D

    # Clip and count
    x_clipped = np.clip(x, -1, 1)
    count = np.sum((x > 1) | (x < -1))

    fs = len(x_clipped) / t
    f, Pxx_den = signal.welch(x_clipped, fs, window='hann', nperseg=int(1e6))

    print(count)
    return f, Pxx_den

In [58]:
def AnalyseFull(maindir, fandir, time):
    main = np.loadtxt(maindir, skiprows=5, delimiter=',', usecols=1)
    if fandir == 'self' or fandir == maindir:
        fan = main
    else:
        fan = np.loadtxt(fandir, skiprows=5, delimiter=',', usecols=1)
    date_match = re.search(r'(\d{2}_\d{2}_\d{4})', maindir)
    if date_match:
        date_str = date_match.group(1)
    else:
        raise ValueError(f"Could not extract date from path: {maindir}")
    filename = os.path.basename(maindir)
    match = re.search(r'_([a-zA-Z0-9]+)_([a-zA-Z0-9]+)--(\d+)\.csv$', filename)
    if match:
        filetype = match.group(1)      # e.g., detectornoise
        config = match.group(2)        # e.g., 5ms
        index = int(match.group(3))    # strip leading zeros
        shortname = f"{date_str}_{filetype}_{config}_{index}"
    else:
        raise ValueError(f"Could not parse shortname from filename: {filename}")
    main_f, main_Pxx = PSDanalysis(main, fan, time)
    np.save(f"{shortname}_f.npy", main_f)
    np.save(f"{shortname}_Pxx.npy", main_Pxx)
    return main_f, main_Pxx

In [62]:
def AnalyseDirs(file_paths,fandir,param_map):
    pattern = re.compile(r'_([a-zA-Z0-9]+)--')
    for path in file_paths:
        filename = os.path.basename(path)
        match = pattern.search(filename)
        if match:
            key = match.group(1)
            if key in param_map:
                params = param_map[key]
                AnalyseFull(path, fandir, **params)
            else:
                print(f"Unknown key '{key}' in file {filename}, skipping.")
        else:
            print(f"Could not extract parameters from {filename}")

In [64]:
def fildirs(mypath):
    onlyfiles = [os.path.join(mypath, f) for f in os.listdir(mypath) if 
    os.path.isfile(os.path.join(mypath, f))]
    return(onlyfiles)